In [13]:
import pandas as pd
import numpy as np
from collections import Counter
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import os
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").values.ravel()
y_test = pd.read_csv("../data/processed/y_test.csv").values.ravel()

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

Train Shape: (1329, 9)
Test Shape: (333, 9)


In [3]:
print("Original Class Distribution:")

print(Counter(y_train))

Original Class Distribution:
Counter({np.int64(0): 884, np.int64(1): 445})


In [4]:
categorical_cols = [
    "ip_type",
    "source"
]

numerical_cols = [
    col for col in X_train.columns
    if col not in categorical_cols
]

In [5]:
preprocessor = ColumnTransformer([

    (
        "num",

        Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]),

        numerical_cols
    ),

    (
        "cat",

        Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="most_frequent")
            ),

            (
                "encoder",

                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ]),

        categorical_cols
    )
])

In [6]:
X_train_processed = preprocessor.fit_transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

print("Processed Train Shape:", X_train_processed.shape)
print("Processed Test Shape:", X_test_processed.shape)

Processed Train Shape: (1329, 11)
Processed Test Shape: (333, 11)


In [7]:
resampling_methods = {

    "Oversampling":
        RandomOverSampler(random_state=42),

    "Undersampling":
        RandomUnderSampler(random_state=42),

    "SMOTE":
        SMOTE(random_state=42),

    "SMOTEENN":
        SMOTEENN(random_state=42)
}

In [8]:
SAVE_PATH = "../data/imbalanced/"

os.makedirs(SAVE_PATH, exist_ok=True)

In [9]:
results = []

for name, sampler in resampling_methods.items():

    print(f"\n🚀 Applying: {name}")

    # Resample
    X_resampled, y_resampled = sampler.fit_resample(
        X_train_processed,
        y_train
    )

    # Check distribution
    print("Distribution:")

    print(Counter(y_resampled))

    # Save datasets
    pd.DataFrame(X_resampled).to_csv(
        SAVE_PATH + f"X_{name}.csv",
        index=False
    )

    pd.DataFrame(y_resampled).to_csv(
        SAVE_PATH + f"y_{name}.csv",
        index=False
    )

    print("✅ Saved")


🚀 Applying: Oversampling
Distribution:
Counter({np.int64(1): 884, np.int64(0): 884})
✅ Saved

🚀 Applying: Undersampling
Distribution:
Counter({np.int64(0): 445, np.int64(1): 445})
✅ Saved

🚀 Applying: SMOTE
Distribution:
Counter({np.int64(1): 884, np.int64(0): 884})
✅ Saved

🚀 Applying: SMOTEENN
Distribution:
Counter({np.int64(0): 859, np.int64(1): 859})
✅ Saved


In [10]:
evaluation_results = []

for name in resampling_methods.keys():

    print(f"\n📊 Evaluating: {name}")

    # Load resampled data
    X_resampled = pd.read_csv(
        SAVE_PATH + f"X_{name}.csv"
    )

    y_resampled = pd.read_csv(
        SAVE_PATH + f"y_{name}.csv"
    ).values.ravel()

    # Train baseline model
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )

    model.fit(X_resampled, y_resampled)

    # Predictions
    y_pred = model.predict(X_test_processed)

    # Metrics
    acc = accuracy_score(y_test, y_pred)

    prec = precision_score(
        y_test,
        y_pred,
        average="macro"
    )

    rec = recall_score(
        y_test,
        y_pred,
        average="macro"
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="macro"
    )

    evaluation_results.append({

        "Technique": name,

        "Accuracy": round(acc, 4),

        "Precision": round(prec, 4),

        "Recall": round(rec, 4),

        "F1-Score": round(f1, 4)
    })

    print("✅ Done")


📊 Evaluating: Oversampling


c:\Users\teler\miniconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


✅ Done

📊 Evaluating: Undersampling


c:\Users\teler\miniconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


✅ Done

📊 Evaluating: SMOTE


c:\Users\teler\miniconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


✅ Done

📊 Evaluating: SMOTEENN
✅ Done


c:\Users\teler\miniconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [11]:
imbalance_results_df = pd.DataFrame(evaluation_results)

imbalance_results_df = imbalance_results_df.sort_values(
    by="F1-Score",
    ascending=False
)

print("\n📊 IMBALANCE COMPARISON RESULTS\n")

print(imbalance_results_df)


📊 IMBALANCE COMPARISON RESULTS

       Technique  Accuracy  Precision  Recall  F1-Score
0   Oversampling       1.0        1.0     1.0       1.0
1  Undersampling       1.0        1.0     1.0       1.0
2          SMOTE       1.0        1.0     1.0       1.0
3       SMOTEENN       1.0        1.0     1.0       1.0


In [12]:
os.makedirs("../reports", exist_ok=True)

imbalance_results_df.to_csv(
    "../reports/imbalance_results.csv",
    index=False
)

print("✅ Results Saved")

✅ Results Saved
